In [0]:
spark.conf.set(
   <YOUR_AZURE_ACCESS_KEY>
)

In [0]:
storage_account_name = "<YOUR_AZURE_STORAGE_ACCOUNT_NAME>"
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/telemetry_data.csv"

raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path)

In [0]:
from pyspark.sql.functions import current_timestamp, col

clean_df = raw_df.dropna() \
    .withColumnRenamed("solar_irradiance_w_m2", "solar_irradiance") \
    .withColumnRenamed("wind_speed_m_s", "wind_speed") \
    .withColumn("ingestion_timestamp", current_timestamp())

In [0]:
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/telemetry_data/"

clean_df.write.format("delta").mode("overwrite").save(silver_path)

In [0]:
display(
    spark.read.format("delta").load(silver_path)
)

In [0]:
spark.read.format("delta").load(silver_path).createOrReplaceTempView("silver_telemetry")